# 01_extract_speech_features

In [10]:
import os
import pandas as pd


# Route setting
base_dir = "/Users/wangtong/Desktop/26 Spring/STAT 605/group_project"
data_dir = os.path.join(base_dir, "download/outputs/merged")
output_dir = os.path.join(base_dir, "analysis/speech_analysis/outputs")

os.makedirs(output_dir, exist_ok=True)

# =========================
# 2. 读取数据
# =========================
msgs = pd.read_parquet(os.path.join(data_dir, "public_messages.parquet"))
players = pd.read_parquet(os.path.join(data_dir, "players.parquet"))
games = pd.read_parquet(os.path.join(data_dir, "games.parquet"))

print("messages:", msgs.shape)
print("players:", players.shape)
print("games:", games.shape)

print("\nColumns in public_messages:")
print(msgs.columns)

# =========================
# 3. player-level speech features
# =========================

# 全部发言
speech_total = (
    msgs.groupby(["game_id", "speaker_id"])
    .agg(
        n_messages=("text", "size"),
        total_text_len=("text_len", "sum"),
        avg_text_len=("text_len", "mean")
    )
    .reset_index()
    .rename(columns={"speaker_id": "player_id"})
)

# 首日发言
speech_day1 = (
    msgs[msgs["day"] == 1]
    .groupby(["game_id", "speaker_id"])
    .agg(
        first_day_messages=("text", "size"),
        first_day_text_len=("text_len", "sum")
    )
    .reset_index()
    .rename(columns={"speaker_id": "player_id"})
)

# 合并到 players
speech_features_by_player = (
    players.merge(speech_total, on=["game_id", "player_id"], how="left")
           .merge(speech_day1, on=["game_id", "player_id"], how="left")
)

# 缺失填0
fill_cols = [
    "n_messages",
    "total_text_len",
    "avg_text_len",
    "first_day_messages",
    "first_day_text_len"
]

for col in fill_cols:
    speech_features_by_player[col] = speech_features_by_player[col].fillna(0)

# 合并 game-level 信息
speech_features_by_player = speech_features_by_player.merge(
    games[["game_id", "winner_team", "last_day", "n_players", "end_reason"]],
    on="game_id",
    how="left"
)

print("\nSpeech features by player:")
print(speech_features_by_player.head())

speech_features_by_player.to_csv(
    os.path.join(output_dir, "speech_features_by_player.csv"),
    index=False
)

# =========================
# 4. game-level speech features
# =========================

game_speech_summary = (
    speech_features_by_player.groupby("game_id")
    .agg(
        total_messages_game=("n_messages", "sum"),
        avg_messages_per_player=("n_messages", "mean"),
        avg_message_length_game=("avg_text_len", "mean"),
        max_messages_one_player=("n_messages", "max")
    )
    .reset_index()
)

game_speech_summary = game_speech_summary.merge(
    games[["game_id", "winner_team", "last_day", "n_players", "end_reason"]],
    on="game_id",
    how="left"
)

print("\nSpeech features by game:")
print(game_speech_summary.head())

game_speech_summary.to_csv(
    os.path.join(output_dir, "speech_features_by_game.csv"),
    index=False
)

print("\nDone. Files saved to:")
print(output_dir)

messages: (40510, 9)
players: (11472, 7)
games: (1435, 6)

Columns in public_messages:
Index(['game_id', 'filename', 'day', 'phase', 'speaker_id', 'event_name',
       'text', 'text_len', 'created_at'],
      dtype='object')

Speech features by player:
    game_id player_id      role               model_name  alive_end  \
0  74788902    Jordan  Werewolf  Grok 4.1 Fast Reasoning      False   
1  74788902     Jamie  Villager                   Grok 4       True   
2  74788902      Alex      Seer                   Grok 4      False   
3  74788902     Casey  Villager   Gemini 3 Flash Preview       True   
4  74788902     Quinn  Werewolf          Claude Opus 4.6      False   

   eliminated_during_day eliminated_during_phase  n_messages  total_text_len  \
0                      2                     Day         4.0          1137.0   
1                     -1                    None         4.0          1005.0   
2                      1                   Night         2.0           662.0   
